# Exploratory Data Analysis of the Vitamin D Transcriptomic Subset

This notebook presents the exploratory data analysis (EDA) of a curated subset of the LINCS L1000 dataset, 
focusing on transcriptional responses to Vitamin D and its analogs in human cell lines. 

The subset was generated from the LINCS2020 release and restricted to:
- **Compounds**: Vitamin D and related analogs (e.g., calcitriol, calcipotriol, paricalcitol, maxacalcitol, ercalcitriol, tacalcitol, seocalcitol).
- **Cell lines**: Five representative human lines (PC3, MCF7, A549, U2OS, HA1E).
- **Perturbation time**: 24 hours.

The aim of this analysis is to:
1. Characterize the distribution of signatures across compounds and cell lines.  
2. Assess the overall structure of the expression matrix.  
3. Explore transcriptomic similarities via dimensionality reduction and clustering.  
4. Evaluate signature quality using available metrics.  

These steps provide the foundation for downstream modeling and biological interpretation.

## Data Loading and Initial Setup

We start by loading the exported subset of the LINCS L1000 dataset, focusing on Vitamin D and its analogs.  
This includes the expression matrix (genes × signatures) and metadata files for signatures, compounds, and cell lines.  
All files are stored in CSV format, curated from the original LINCS2020 release.

In [ ]:
# Import core libraries
import os
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Write Parquet using pyarrow directly (preserves the gene index)
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
# Load subset metadata and expression files
cell_meta = pd.read_csv("../data/exports/subset_cell_lines_meta.csv")
cmp_meta = pd.read_csv("../data/exports/subset_compounds_meta.csv")
gene_meta = pd.read_csv("../data/exports/subset_genes_meta.csv")
sig_meta = pd.read_csv("../data/exports/subset_signatures_meta.csv")
exp_df = pd.read_csv("../data/exports/subset_expression_wide_gene_id.csv", index_col=0)

In [ ]:
# Quick checks on dimensions
print("Cell lines:", cell_meta.shape)
print("Compounds:", cmp_meta.shape)
print("Genes:", gene_meta.shape)
print("Signatures:", sig_meta.shape)
print("Expression matrix:", exp_df.shape)

# Preview first rows
display(cell_meta.head())
display(cmp_meta.head())
display(sig_meta.head())
display(exp_df.iloc[:5, :5])


# Utility function for quick dataframe check
def quick_check(df, name, n=3):
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("Null values:\n", df.isna().sum().sort_values(ascending=False).head())
    print("Duplicates:", df.duplicated().sum())
    print("First rows:")
    display(df.head(n))
    print("\n")

# Run checks on metadata
quick_check(cell_meta, "Cell metadata")
quick_check(cmp_meta, "Compound metadata")
quick_check(gene_meta, "Gene metadata")
quick_check(sig_meta, "Signature metadata")

# Special check for expression matrix
print("--- Expression matrix ---")
print("Shape:", exp_df.shape)
print("Nulls (per signature):", (exp_df.isna().sum(axis=0) > 0).sum(), "/", exp_df.shape[1])
print("Nulls (per gene):", (exp_df.isna().sum(axis=1) > 0).sum(), "/", exp_df.shape[0])
display(exp_df.iloc[:5, :5])

### Sanity Check Results

- **Cell metadata (5×5)**  
  - One missing value in `growth_pattern` (HA1E).  
  - No duplicates. → ✔️ clean.  

- **Compound metadata (12×7)**  
  - All entries in `compound_aliases` are missing (NaN in all 12 rows).  
  - The rest is complete. → ✔️ usable; this column can be ignored if not needed.  

- **Gene metadata (12,328×7)**  
  - 51 genes without `ensembl_id`.  
  - The rest is complete, no duplicates. → ✔️ clean, except for minor missing annotations.  

- **Signature metadata (422×14)**  
  - No missing values.  
  - No duplicates. → ✔️ perfect.  

- **Expression matrix (12,328×424)**  
  - 164 signatures entirely empty (all values NaN).  
  - Consequently, all 12,328 genes show NaNs in those signatures.  
  - Conclusion: there are **422 metadata entries for signatures** but **only 258 with actual expression data**.  


#### **Decision Taken**

- Remove the 164 empty signatures from the expression matrix, keeping the 258 valid ones.  
- Align `sig_meta` to retain only the corresponding 258 signatures.  
- Keep all other metadata tables (cell, compound, gene) unchanged.  


In [ ]:
### Cleaning step: remove empty signatures and realign metadata

# --- Diagnose missing values ---
exp_numeric = exp_df.select_dtypes(include=[np.number])  # genes × signatures

nan_by_sig = exp_numeric.isna().sum(axis=0).sort_values(ascending=False)  # per signature (column)
nan_by_gene = exp_numeric.isna().sum(axis=1).sort_values(ascending=False) # per gene (row)

# Identify signatures with no NaNs (valid signatures)
valid_sig_cols = nan_by_sig[nan_by_sig == 0].index.tolist()

# Filter expression matrix to keep only valid signatures
exp_clean = exp_numeric[valid_sig_cols]   # shape: genes × valid_signatures
print("Expression matrix shape after cleaning:", exp_clean.shape)

# Align signature metadata to the cleaned matrix
sig_meta_clean = (
    sig_meta.set_index('sig_id')
    .reindex(valid_sig_cols)
    .reset_index()               # bring sig_id back as a column
    .rename(columns={"index":"sig_id"})  # ensure column name is consistent
)

# Sanity check
assert list(valid_sig_cols) == list(sig_meta_clean['sig_id'])
print("Signature metadata shape after cleaning:", sig_meta_clean.shape)

## Signature Distribution by Compound and Cell Line — setup

We quantify how many signatures are available per compound and per cell line to detect potential imbalance that could bias downstream analyses.

In [ ]:
# Build tidy summary tables for signatures per compound and per cell line

# Map pert_id -> compound name
pert_lookup = cmp_meta[['pert_id', 'cmap_name']].drop_duplicates()

# --- Counts by compound (pert_id) ---
sig_by_pert = (
    sig_meta
    .groupby('pert_id', as_index=False)
    .agg(n_signatures=('sig_id', 'count'))
    .merge(pert_lookup, on='pert_id', how='left')
    .assign(cmap_name=lambda d: d['cmap_name'].fillna(d['pert_id']))
    .sort_values('n_signatures', ascending=False)
    .reset_index(drop=True)
)

# Count signatures grouped by compound name (cmap_name)
sig_by_cmap = (
    sig_meta
    .merge(cmp_meta[['pert_id', 'cmap_name']], on='pert_id', how='left')
    .groupby('cmap_name', as_index=False)
    .agg(n_signatures=('sig_id', 'count'))
    .sort_values('n_signatures', ascending=False)
    .reset_index(drop=True)
)

display(sig_by_cmap)


# --- Counts by cell line ---
sig_by_cell = (
    sig_meta
    .groupby('cell_id', as_index=False)
    .agg(n_signatures=('sig_id', 'count'))
    .sort_values('n_signatures', ascending=False)
    .reset_index(drop=True)
)

# Display summary tables
display(sig_by_pert)
display(sig_by_cell)


In [ ]:
# Create side-by-side barplots: compound vs. cell line
fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={'width_ratios':[2,1]})

# --- Left: signatures per compound (cmap_name) ---
sns.barplot(
    data=sig_by_cmap,
    x='n_signatures',
    y='cmap_name',
    order=sig_by_cmap.sort_values('n_signatures', ascending=False)['cmap_name'],
    errorbar=None,
    ax=axes[0],
    color="steelblue"
)
axes[0].set_xlabel('Number of signatures')
axes[0].set_ylabel('Compound (cmap_name)')
axes[0].set_title('Signatures per compound')

# --- Right: signatures per cell line ---
sns.barplot(
    data=sig_by_cell,
    x='cell_id',
    y='n_signatures',
    order=sig_by_cell.sort_values('n_signatures', ascending=False)['cell_id'],
    errorbar=None,
    ax=axes[1],
    color="steelblue"
)
axes[1].set_xlabel('Cell line')
axes[1].set_ylabel('Number of signatures')
axes[1].set_title('Signatures per cell line')

plt.tight_layout()
plt.show()


#### Conclusion

The distribution of signatures is uneven across compounds and cell lines.  
**Calcitriol** is the most represented compound (115 signatures), followed by **maxacalcitol** (60) and several analogs with ~57 signatures. **Calcipotriol** is the least represented (31).  

Across cell lines, **MCF7** and **A549** show the highest coverage (>100 signatures each), while **U2OS** is markedly underrepresented (21 signatures).  
This imbalance should be considered in downstream analyses to avoid biases in compound- or cell line–specific conclusions.

---

## Global Distribution of Expression Values

To evaluate the overall structure of the dataset, we inspect the distribution of moderated `z-scores` across all genes and signatures.  
This step helps to:  
- Assess the expected centering around zero.  
- Verify the range and spread of values.  
- Identify potential outliers that may influence downstream analyses.


In [ ]:
# Use only numeric expression columns (signatures)
exp_numeric = exp_df.select_dtypes(include=[np.number])

# Flatten to 1D numeric array
vals = exp_numeric.to_numpy().ravel()

# Create side-by-side panels: histogram and boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios':[3,1]})

# --- Left: histogram ---
sns.histplot(x=vals, bins=100, kde=True, ax=axes[0], color="green")
axes[0].set_xlabel("Z-score value")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of expression values")

# --- Right: boxplot ---
sns.boxplot(y=vals, ax=axes[1], color="yellow")
axes[1].set_ylabel("Z-score value")
axes[1].set_title("Global distribution (boxplot)")

plt.tight_layout()
plt.show()

#### Conclusion

The global distribution of expression values is tightly centered around zero, with most `z-scores` within the expected range of ±3.  
Both histogram and boxplot highlight a small proportion of extreme values, which are typical for high-dimensional transcriptomic data and do not indicate systematic anomalies.

---

## Validation of Perturbation Time and Dose

Although the subset was filtered to 24 hours, we verify the consistency of perturbation times directly from the metadata.  
We also inspect the distribution of applied doses across compounds, as variations in concentration may contribute to heterogeneity in transcriptional responses.


In [ ]:
# Check unique perturbation times
print("Unique perturbation times:", sig_meta['pert_time'].unique())
print("Unique time units:", sig_meta['pert_time_unit'].unique())

# Summarize doses by compound
dose_summary = (
    sig_meta
    .merge(cmp_meta[['pert_id','cmap_name']], on='pert_id', how='left')
    .groupby(['cmap_name','pert_dose','pert_dose_unit'], as_index=False)
    .agg(n_signatures=('sig_id','count'))
    .sort_values(['cmap_name','pert_dose'])
)

display(dose_summary)

plt.figure(figsize=(10,5))
sns.boxplot(
    data=sig_meta.merge(cmp_meta[['pert_id','cmap_name']], on='pert_id', how='left'),
    x='cmap_name',
    y='pert_dose',
    hue='cmap_name',         # use cmap_name as hue
    dodge=False,             # avoid side-by-side duplication
    legend=False,            # hide redundant legend
    palette="Set3"           # color palette
)
plt.yscale('log')
plt.xlabel("Compound (cmap_name)")
plt.ylabel("Dose (µM, log scale)")
plt.title("Dose distribution across compounds")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### Conclusion

All perturbation times are consistently set to 24h, confirming the filtering criteria.  
Dose distributions span several orders of magnitude (from ~0.01 µM to 10 µM), with some compounds showing broader coverage (e.g., calcitriol, calcipotriol) while others are more restricted.  
This heterogeneity in dosing conditions may contribute to variability in transcriptional responses and should be taken into account in downstream analyses.

---

## Principal Component Analysis (PCA)

To explore global similarities among signatures, we apply `Principal Component Analysis` (`PCA`) on the expression matrix.  
This method reduces the dimensionality of the dataset while retaining as much variance as possible, allowing us to visualize whether signatures cluster by compound or cell line.

In [ ]:
# Transpose: signatures × genes
X = exp_clean.T  

# Scale data
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X)

# Fit PCA (2 components for visualization)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Build results dataframe
pca_df = pd.DataFrame(X_pca, columns=['PC1','PC2'])
pca_df = pd.concat(
    [pca_df, sig_meta_clean[['sig_id','cell_id','pert_id']].reset_index(drop=True)],
    axis=1
).merge(cmp_meta[['pert_id','cmap_name']], on='pert_id', how='left')

# PCA plots: compounds vs cell lines, side by side
fig, axes = plt.subplots(1, 2, figsize=(14,6))

# --- Left: PCA by compound ---
sns.scatterplot(
    data=pca_df, x='PC1', y='PC2',
    hue='cmap_name', palette='Set2', s=60, alpha=0.85,
    ax=axes[0]
)
axes[0].set_title("PCA of Vitamin D signatures (by compound)")

# --- Right: PCA by cell line ---
sns.scatterplot(
    data=pca_df, x='PC1', y='PC2',
    hue='cell_id', palette='tab10', s=60, alpha=0.85,
    ax=axes[1]
)
axes[1].set_title("PCA of Vitamin D signatures (by cell line)")

plt.tight_layout()
plt.show()

# Variance explained
print("Explained variance (PC1, PC2):", pca.explained_variance_ratio_)


#### Conclusion

The PCA projection of the 258 valid signatures shows that the first two components explain **13.6%** and **5.4%** of the total variance, respectively.  
No strong global separation is observed among `compounds`, suggesting largely overlapping transcriptional profiles across Vitamin D analogs.  
When colored by `cell line`, mild clustering tendencies appear, particularly for one lineage, indicating that **cellular context contributes more strongly to variance structure than compound identity**.  
These results are consistent with the high-dimensional nature of transcriptomic data, where many components are needed to capture the full complexity of variation.

## Scree Plot: Explained Variance of Principal Components

To evaluate how much variance is captured by each principal component (PC), we generated a scree plot.  
This allows us to determine how many PCs contribute meaningfully to the variance structure of the dataset, guiding dimensionality reduction choices.

**Decision taken:**  
The explained variance drops quickly after the first components, confirming that only a limited number of PCs capture a substantial fraction of the variation. For subsequent analyses, we will retain the first components up to the "elbow point" of the scree plot.


In [ ]:
# Run PCA on the expression matrix (transpose so samples are rows)
pca = PCA()
pca.fit(exp_clean.T)

# Explained variance
explained_var = pca.explained_variance_ratio_ * 100  # in %
cumulative_var = np.cumsum(explained_var)

# Scree plot
plt.figure(figsize=(8,6))
plt.plot(range(1, len(explained_var)+1), explained_var, marker='o', label="Individual variance")
plt.plot(range(1, len(explained_var)+1), cumulative_var, marker='s', linestyle='--', label="Cumulative variance")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance (%)")
plt.title("Scree Plot of PCA")
plt.legend()
plt.tight_layout()
plt.show()


### Conclusion — Scree Plot

The scree plot shows a steep drop in explained variance after the first few components, followed by a long tail.  
This indicates that only a limited number of principal components capture a substantial share of the total variance, while many additional components contribute marginally.

Based on variance thresholds (see code output below), we will retain up to the elbow/threshold identified for downstream summaries, and move targeted analyses (e.g., dose comparison, ANOVA on PCs, enrichment) to a separate notebook.


In [ ]:
# Quantify how many PCs are needed to reach common variance thresholds
pca_full = PCA().fit(exp_clean.T)
explained = pca_full.explained_variance_ratio_
cum = np.cumsum(explained)

def pcs_for(threshold):
    return int(np.searchsorted(cum, threshold) + 1)

for thr in [0.50, 0.70, 0.80, 0.90, 0.95]:
    print(f"PCs to reach {int(thr*100)}% variance:", pcs_for(thr))


### PCA Variance Explained

The PCA variance analysis indicates that:

- ~27 PCs are required to capture **50%** of the variance.  
- ~62 PCs are required to capture **70%** of the variance.  
- ~91 PCs are required to capture **80%** of the variance.  
- ~138 PCs are required to capture **90%** of the variance.  
- ~177 PCs are required to capture **95%** of the variance.  

This distribution confirms the high-dimensional nature of transcriptomic data, where variance is spread across many axes. While no single component dominates, the first ~50–100 PCs already summarize a large portion of the signal and are suitable for downstream association tests (e.g., ANOVA with metadata, dose-response contrasts).


### Closing & Export of Cleaned Data

We export the cleaned datasets in two complementary formats:

- **Expression matrix (`exp_clean`)**: saved as **Parquet** to preserve the gene index efficiently and enable fast I/O in downstream analyses.  
- **Signature metadata (`sig_meta_clean`)**: saved as **CSV**, keeping only the data columns (without index), since `sig_id` is the primary key.  

This ensures reproducibility and consistency when reloading the data in subsequent notebooks.

In [ ]:
# Create output folder if needed
out_dir = "../data/exports"
os.makedirs(out_dir, exist_ok=True)

# ---- Expression matrix (genes × signatures) ----
# Ensure column order matches metadata and set a clear index name
exp_clean = exp_clean.loc[:, sig_meta_clean['sig_id']]
exp_clean.index.name = 'gene_id'

table = pa.Table.from_pandas(exp_clean, preserve_index=True)
pq.write_table(table, f"{out_dir}/expression_matrix_clean.parquet", compression="snappy")

# ---- Signature metadata (aligned to exp_clean columns) ----
sig_meta_clean.to_csv(f"{out_dir}/signature_metadata_clean.csv", index=False)

print("✅ Cleaned data exported:",
      "\n- expression_matrix_clean.parquet",
      "\n- signature_metadata_clean.csv")